# TESSERA 1B-M VAE-latent variant shortlist

**Goal**: compare the new VAE latents built on TESSERA v2 **1B-M** embeddings
(8 variants: crop {64,128} x latent dim {16,32,64} x aux {on,off}, for embedding
years **2017** and **2024**) against the **previous default** (old TESSERA v1,
`vae_lat16` +mTPI) and the no-TESSERA references — to shortlist which variant to
roll out to the remaining regions.

All runs share the paper config (direct concat, +mTPI, with elevation, no static
fields, wd=1e-4) and differ ONLY in the latents file. Folders:

| axis | where it lives |
|---|---|
| region (EU / East Asia) + year (2017/2024) | experiment folder `snapshot_14y_<region>_tessera_1B-M_<year>` |
| crop / lat dim / aux | experiment name inside the folder |

**Re-run top to bottom any time** — cells are robust to partially-finished sweeps
(pending runs simply show as missing seeds).

Caveats to keep in mind when reading:

- **Station sets**: new runs use old-patch-filter ∩ new-latent-valid. vs the old
  default runs: EU −10/+1 stations (held-out 1758 → 1755), EA +1 — negligible but
  not zero. 2017 vs 2024 sets also differ by ≤10 stations.
- `p128_2024_crop128_lat32_auxoff` VAE was still training at sweep launch — its
  2024 runs lag the rest (auto-picked-up once submitted).
- Wind point MAE is MAE@median (truncated-normal convention); t2m is mean-based
  Gaussian MAE. The `point_mae` / `point_rmse` canonical columns handle this.
  NLL / CRPS are proper scoring rules, comparable across heads.

> **Phase 2 (2026-07-20)**: after the EU/EA round eliminated crop128 and lat64, the crop64 shortlist (lat {16,32} x aux {on,off}, both years) was rolled out to US / Australia / Southern Africa — 144 jobs. The skill tables aggregate over every complete (region x target) cell; `n_cells` shows how many each candidate has (10 = all regions).

> **Archive note:** only the `_tessera_1B-M_2017` and `_2017_shuffled` experiment
> folders ship in `scripts/experiments/`; the `_tessera_1B-M_2024` folders exist as
> `training_runs_*` output directories only, so the 2024 half of this comparison
> cannot be loaded here (`load_folder_results` needs the folder's
> `experiments.yaml`). Retained as model-selection provenance.


In [ ]:
# ---- Setup: constants + load ----------------------------------------------
import re

import _helpers as H
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

pd.set_option("display.width", 200)

SEEDS = (42, 123, 456)
REGIONS = ["europe", "east_asia", "us", "australia", "southern_africa"]
YEARS = [2017, 2024]
TARGETS = ["t2m", "wind"]

# (region, year) -> new sweep folder
FOLDERS_NEW = {
    ("europe", 2017): "snapshot_14y_eu_tessera_1B-M_2017",
    ("europe", 2024): "snapshot_14y_eu_tessera_1B-M_2024",
    ("east_asia", 2017): "snapshot_14y_east_asia_tessera_1B-M_2017",
    ("east_asia", 2024): "snapshot_14y_east_asia_tessera_1B-M_2024",
    # Phase-2 rollout: crop64 shortlist only (lat {16,32} x aux {on,off}).
    ("us", 2017): "snapshot_14y_us_tessera_1B-M_2017",
    ("us", 2024): "snapshot_14y_us_tessera_1B-M_2024",
    ("australia", 2017): "snapshot_14y_australia_tessera_1B-M_2017",
    ("australia", 2024): "snapshot_14y_australia_tessera_1B-M_2024",
    ("southern_africa", 2017): "snapshot_14y_southern_africa_tessera_1B-M_2017",
    ("southern_africa", 2024): "snapshot_14y_southern_africa_tessera_1B-M_2024",
}
# region -> folder holding the previous default + references
FOLDERS_OLD = {
    "europe": "snapshot_14y_eu",
    "east_asia": "snapshot_14y_east_asia",
    "us": "snapshot_14y_us",
    "australia": "snapshot_14y_australia",
    "southern_africa": "snapshot_14y_southern_africa",
}

# Reference experiments (paper +mTPI table): previous default TESSERA v1
# lat16, its trained no-TESSERA twin, and the non-trained references.
REF_EXPS = {
    "t2m": {
        "old default (v1 lat16)": "t2m_snap_vae_lat16_concat_with_elev_mtpi_no_static_wd",
        "no-TESSERA ConvCNP": "t2m_snap_bilinear_baseline_mtpi_wd",
        "ERA5-interp": "t2m_snap_era5_interp_baseline",
        "persistence": "t2m_snap_persistence_baseline",
    },
    "wind": {
        "old default (v1 lat16)": "wind_truncnormal_snap_vae_lat16_concat_with_elev_mtpi_no_static_wd",
        "no-TESSERA ConvCNP": "wind_truncnormal_snap_bilinear_baseline_mtpi_wd",
        "ERA5-interp": "wind_snap_era5_interp_baseline",
        "persistence": "wind_snap_persistence_baseline",
    },
}

# Display order of the 8 swept variants (crop, then lat, then aux).
VARIANT_ORDER = [
    "crop64_lat16_auxoff",
    "crop64_lat16_auxon",
    "crop64_lat32_auxoff",
    "crop64_lat32_auxon",
    "crop128_lat16_auxoff",
    "crop128_lat32_auxoff",
    "crop128_lat64_auxoff",
    "crop128_lat64_auxon",
]

_VAR_RE = re.compile(r"crop(?P<crop>\d+)_lat(?P<lat>\d+)_aux(?P<aux>on|off)")


def parse_variant(exp_name):
    """Extract (variant_key, crop, lat, aux, target) from a sweep entry name."""
    m = _VAR_RE.search(exp_name)
    if m is None:
        return None
    target = "t2m" if exp_name.startswith("t2m") else "wind"
    return dict(
        variant=f"crop{m['crop']}_lat{m['lat']}_aux{m['aux']}",
        crop=int(m["crop"]),
        lat=int(m["lat"]),
        aux=m["aux"],
        target=target,
    )


# ---- Load new-sweep folders ----
new_frames = []
for (region, year), folder in FOLDERS_NEW.items():
    d = H.load_folder_results(folder, seeds=SEEDS)
    if d.empty:
        continue
    d["region"], d["year"] = region, year
    new_frames.append(d)
df_new = pd.concat(new_frames, ignore_index=True) if new_frames else pd.DataFrame()
if not df_new.empty:
    axes = df_new["experiment"].apply(parse_variant)
    for k in ("variant", "crop", "lat", "aux", "target"):
        df_new[k] = axes.apply(lambda a: a[k] if a else None)

# ---- Load reference folders (previous default + baselines) ----
old_frames = []
for region, folder in FOLDERS_OLD.items():
    d = H.load_folder_results(folder, seeds=SEEDS)
    if d.empty:
        continue
    d["region"] = region
    old_frames.append(d)
df_old = pd.concat(old_frames, ignore_index=True) if old_frames else pd.DataFrame()
if not df_old.empty:
    keep = {n: (t, lbl) for t, d in REF_EXPS.items() for lbl, n in d.items()}
    df_old = df_old[df_old["experiment"].isin(keep)].copy()
    df_old["target"] = df_old["experiment"].map(lambda n: keep[n][0])
    df_old["ref_label"] = df_old["experiment"].map(lambda n: keep[n][1])

print(f"new-sweep rows loaded: {len(df_new)}   reference rows loaded: {len(df_old)}")

In [ ]:
# ---- Completeness: which runs have finished? -------------------------------
# Expected: 8 variants x 2 targets x 3 seeds per folder (the pending 2024
# crop128_lat32_auxoff VAE shows up here as missing until its jobs run).
rows = []
for (region, year), folder in FOLDERS_NEW.items():
    sub = (
        df_new[(df_new.get("region") == region) & (df_new.get("year") == year)]
        if not df_new.empty
        else pd.DataFrame()
    )
    for v in VARIANT_ORDER:
        cell = {}
        for t in TARGETS:
            n = (
                0
                if sub.empty
                else int(((sub["variant"] == v) & (sub["target"] == t)).sum())
            )
            cell[t] = n
        rows.append(
            {
                "folder": folder,
                "variant": v,
                "t2m": f"{cell['t2m']}/3",
                "wind": f"{cell['wind']}/3",
                "done": cell["t2m"] + cell["wind"],
            }
        )
comp = pd.DataFrame(rows)
total_done = int(comp["done"].sum())
print(
    f"finished runs: {total_done}. Gaps are intentional: crop128_lat32_auxoff-2024 "
    f"(EU/EA) was cancelled after crop128 was eliminated, and US/AUS/SA run only "
    f"the crop64 shortlist (phase 2)."
)
display(
    comp.pivot(index="variant", columns="folder", values="t2m")
    .reindex(VARIANT_ORDER)
    .rename_axis(None, axis=1)
)
display(
    comp.pivot(index="variant", columns="folder", values="wind")
    .reindex(VARIANT_ORDER)
    .rename_axis(None, axis=1)
)

In [ ]:
# ---- Aggregation helpers ---------------------------------------------------
METRICS = ["point_mae", "point_rmse", "nll", "crps"]


def agg_rows(rows, target):
    """mean/std/n per metric over seed rows, for one (experiment, cell)."""
    out = {}
    for m in METRICS:
        mean, std, n = H._agg_metric(rows, f"{target}_{m}")
        out[m], out[f"{m}_std"] = mean, std
        if m == "point_mae":
            out["n_seeds"] = n
    return out


def fmt(mean, std, prec=3):
    return "—" if mean != mean else f"{mean:.{prec}f}±{std:.{prec}f}"


def ref_agg(region, target, label):
    """Aggregate one reference experiment for a (region, target) cell."""
    if df_old.empty:
        return {m: float("nan") for m in METRICS} | {"n_seeds": 0}
    rows = df_old[
        (df_old["region"] == region)
        & (df_old["target"] == target)
        & (df_old["ref_label"] == label)
    ]
    return agg_rows(rows, target)


def variant_agg(region, year, variant, target):
    if df_new.empty:
        return {m: float("nan") for m in METRICS} | {"n_seeds": 0}
    rows = df_new[
        (df_new["region"] == region)
        & (df_new["year"] == year)
        & (df_new["variant"] == variant)
        & (df_new["target"] == target)
    ]
    return agg_rows(rows, target)

In [ ]:
# ---- Headline tables: one per (region x target) ----------------------------
# Rows: non-trained refs, trained no-TESSERA baseline, previous default, then
# the 16 new (year x variant) combinations. Lower is better everywhere.
for region in REGIONS:
    for target in TARGETS:
        rows = []
        for lbl in [
            "persistence",
            "ERA5-interp",
            "no-TESSERA ConvCNP",
            "old default (v1 lat16)",
        ]:
            a = ref_agg(region, target, lbl)
            rows.append(
                {
                    "model": lbl,
                    "year": "—",
                    "MAE": fmt(a["point_mae"], a["point_mae_std"]),
                    "RMSE": fmt(a["point_rmse"], a["point_rmse_std"]),
                    "NLL": fmt(a["nll"], a["nll_std"]),
                    "CRPS": fmt(a["crps"], a["crps_std"]),
                    "seeds": a["n_seeds"],
                }
            )
        for year in YEARS:
            for v in VARIANT_ORDER:
                a = variant_agg(region, year, v, target)
                if a["n_seeds"] == 0:
                    continue
                rows.append(
                    {
                        "model": f"1B-M {v}",
                        "year": year,
                        "MAE": fmt(a["point_mae"], a["point_mae_std"]),
                        "RMSE": fmt(a["point_rmse"], a["point_rmse_std"]),
                        "NLL": fmt(a["nll"], a["nll_std"]),
                        "CRPS": fmt(a["crps"], a["crps_std"]),
                        "seeds": a["n_seeds"],
                    }
                )
        print(
            f"\n=== {region} — {target} "
            f"({'Gaussian, mean-MAE' if target == 't2m' else 'TruncNormal, MAE@median'}) ==="
        )
        display(pd.DataFrame(rows).set_index(["model", "year"]))

In [ ]:
# ---- Skill vs the trained no-TESSERA baseline + shortlist ranking ----------
# skill = 100 * (baseline_metric - model_metric) / baseline_metric  (higher =
# better; positive = beats the no-TESSERA ConvCNP). Computed per (region x
# target) cell on seed MEANS, for MAE and CRPS. The shortlist ranks (year,
# variant) by mean MAE-skill across all complete cells; `min_skill` and
# `n_cells` guard against a variant that only looks good where results
# happen to have finished.
def skill_table(metric="point_mae"):
    recs = []
    base = {
        (r, t): ref_agg(r, t, "no-TESSERA ConvCNP")[metric]
        for r in REGIONS
        for t in TARGETS
    }
    candidates = [("old", None, "old default (v1 lat16)")] + [
        (year, v, f"1B-M {v}") for year in YEARS for v in VARIANT_ORDER
    ]
    for year, v, label in candidates:
        rec = {"model": label, "year": year if year != "old" else "—"}
        skills = []
        for r in REGIONS:
            for t in TARGETS:
                mv = (
                    ref_agg(r, t, "old default (v1 lat16)")[metric]
                    if year == "old"
                    else variant_agg(r, year, v, t)[metric]
                )
                b = base[(r, t)]
                s = 100 * (b - mv) / b if (mv == mv and b == b) else float("nan")
                rec[f"{r[:2]}_{t}"] = round(s, 2) if s == s else float("nan")
                if s == s:
                    skills.append(s)
        rec["mean_skill"] = round(float(np.mean(skills)), 2) if skills else float("nan")
        rec["min_skill"] = round(float(np.min(skills)), 2) if skills else float("nan")
        rec["n_cells"] = len(skills)
        recs.append(rec)
    return (
        pd.DataFrame(recs)
        .sort_values("mean_skill", ascending=False, na_position="last")
        .set_index(["model", "year"])
    )


print("MAE skill vs no-TESSERA ConvCNP (+ = better than baseline), % :")
mae_rank = skill_table("point_mae")
display(mae_rank)
print("CRPS skill vs no-TESSERA ConvCNP, % :")
display(skill_table("crps"))
complete = mae_rank[mae_rank["n_cells"] == 4]
if not complete.empty:
    best = complete.index[0]
    print(
        f"\nCurrent shortlist leader (complete cells only): {best[0]} "
        f"[{best[1]}]  mean={complete.iloc[0]['mean_skill']}% "
        f"min={complete.iloc[0]['min_skill']}%"
    )
else:
    print("\nNo (year, variant) has all 4 region x target cells complete yet.")

In [ ]:
# ---- Dot plots: metric per variant, seeds shown, refs as lines -------------
# Identity by y-position (variant), colour only distinguishes embedding year.
# Validated palette: 2017 = blue #2a78d6, 2024 = orange #eb6834.
YEAR_COLOUR = {2017: "#2a78d6", 2024: "#eb6834"}
_REF_STYLE = {  # line style + grey ink for the reference lines
    "old default (v1 lat16)": dict(ls="--", color="#0b0b0b"),
    "no-TESSERA ConvCNP": dict(ls="-", color="#52514e"),
    "ERA5-interp": dict(ls=":", color="#9a988f"),
}


def dot_plot(metric="point_mae", label="MAE"):
    fig, axes_ = plt.subplots(
        len(REGIONS),
        len(TARGETS),
        figsize=(12, 3 + 0.42 * len(VARIANT_ORDER)),
        sharey=True,
    )
    ypos = {v: i for i, v in enumerate(VARIANT_ORDER[::-1])}
    for i, region in enumerate(REGIONS):
        for j, target in enumerate(TARGETS):
            ax = axes_[i][j] if len(REGIONS) > 1 else axes_[j]
            for year in YEARS:
                dy = -0.17 if year == 2024 else 0.17
                for v in VARIANT_ORDER:
                    if df_new.empty:
                        continue
                    rows = df_new[
                        (df_new["region"] == region)
                        & (df_new["year"] == year)
                        & (df_new["variant"] == v)
                        & (df_new["target"] == target)
                    ]
                    vals = pd.to_numeric(
                        rows.get(f"{target}_{metric}"), errors="coerce"
                    ).dropna()
                    if vals.empty:
                        continue
                    y = ypos[v] + dy
                    ax.plot(
                        vals,
                        [y] * len(vals),
                        "o",
                        ms=3.5,
                        alpha=0.45,
                        color=YEAR_COLOUR[year],
                        zorder=2,
                    )
                    ax.plot(
                        [vals.mean()],
                        [y],
                        "o",
                        ms=8,
                        color=YEAR_COLOUR[year],
                        zorder=3,
                        label=str(year) if (v == VARIANT_ORDER[0]) else None,
                    )
            for lbl, style in _REF_STYLE.items():
                a = ref_agg(region, target, lbl)
                if a[metric] == a[metric]:
                    ax.axvline(a[metric], lw=1.4, zorder=1, **style)
            ax.set_yticks(range(len(VARIANT_ORDER)))
            ax.set_yticklabels(VARIANT_ORDER[::-1], fontsize=8.5)
            ax.set_title(f"{region} — {target}", fontsize=10)
            ax.grid(axis="x", color="#e8e7e0", lw=0.8)
            ax.set_axisbelow(True)
            for s in ("top", "right"):
                ax.spines[s].set_visible(False)
            if i == len(REGIONS) - 1:
                ax.set_xlabel(label, fontsize=9)
    handles, labels_ = (
        axes_[0][0] if len(REGIONS) > 1 else axes_[0]
    ).get_legend_handles_labels()
    ref_handles = [plt.Line2D([], [], lw=1.4, **s) for s in _REF_STYLE.values()]
    fig.legend(
        handles + ref_handles,
        labels_ + list(_REF_STYLE),
        loc="upper center",
        bbox_to_anchor=(0.5, 1.045),
        ncol=5,
        frameon=False,
        fontsize=9,
    )
    fig.suptitle(
        f"TESSERA 1B-M variants — {label} (dots = seeds, big dot = mean; "
        f"lower is better)",
        y=1.09,
        fontsize=11,
    )
    fig.tight_layout()
    plt.show()


dot_plot("point_mae", "point MAE")
dot_plot("crps", "CRPS")

In [ ]:
# ---- Axis marginals: which sweep axis actually matters? --------------------
# Mean MAE-skill (vs no-TESSERA baseline) over all complete (region x target
# x variant) cells sharing each axis level. Read as: "holding everything else
# mixed, moving this axis to level X is worth ~Y % MAE". Error bar = std over
# the contributing cells (spread, not a CI).
AXES_DEF = {"year": YEARS, "crop": [64, 128], "lat": [16, 32, 64], "aux": ["off", "on"]}


def marginal_skills():
    base = {
        (r, t): ref_agg(r, t, "no-TESSERA ConvCNP")["point_mae"]
        for r in REGIONS
        for t in TARGETS
    }
    recs = []
    for year in YEARS:
        for v in VARIANT_ORDER:
            pv = parse_variant(f"t2m_{v}")  # crop/lat/aux from the key
            for r in REGIONS:
                for t in TARGETS:
                    mv = variant_agg(r, year, v, t)["point_mae"]
                    b = base[(r, t)]
                    if mv == mv and b == b:
                        recs.append(
                            {
                                "year": year,
                                "crop": pv["crop"],
                                "lat": pv["lat"],
                                "aux": pv["aux"],
                                "skill": 100 * (b - mv) / b,
                            }
                        )
    return pd.DataFrame(recs)


msk = marginal_skills()
if msk.empty:
    print("No complete cells yet — marginals appear once results land.")
else:
    fig, axs = plt.subplots(1, len(AXES_DEF), figsize=(11, 2.8), sharex=True)
    for ax, (axis, levels) in zip(axs, AXES_DEF.items(), strict=False):
        stats = [(lv, msk[msk[axis] == lv]["skill"]) for lv in levels]
        for k, (lv, vals) in enumerate(stats):
            if vals.empty:
                continue
            ax.errorbar(
                vals.mean(),
                k,
                xerr=vals.std(),
                fmt="o",
                ms=7,
                color="#2a78d6",
                ecolor="#9ec5f4",
                capsize=3,
                lw=1.6,
            )
            ax.annotate(
                f" n={len(vals)}",
                (vals.mean(), k),
                fontsize=7.5,
                color="#52514e",
                xytext=(4, 5),
                textcoords="offset points",
            )
        ax.set_yticks(range(len(levels)))
        ax.set_yticklabels([str(l) for l in levels], fontsize=9)
        ax.set_title(axis, fontsize=10)
        ax.grid(axis="x", color="#e8e7e0", lw=0.8)
        ax.set_axisbelow(True)
        for s in ("top", "right"):
            ax.spines[s].set_visible(False)
    axs[0].set_xlabel("mean MAE skill vs baseline (%)", fontsize=9)
    fig.suptitle(
        "Marginal effect of each sweep axis (pooled over everything else)", fontsize=11
    )
    fig.tight_layout()
    plt.show()
    display(msk.groupby("year")["skill"].agg(["mean", "std", "count"]).round(2))

## Reading the results / next steps

1. **Shortlist rule of thumb**: prefer the (year, variant) with the best
   `mean_skill` **among those with `n_cells == 4`** and a `min_skill` that
   doesn't collapse in any cell — a variant that wins on average but loses to
   the old default somewhere is a risky rollout choice. Cross-check the CRPS
   ranking: a good variant should win on both point accuracy and the proper
   scoring rule.
2. **Old default as the bar**: the previous default (v1 lat16 +mTPI) row in the
   skill table is the number to beat — if no 1B-M variant clears it in both
   regions, the new embeddings don't justify a rollout yet.
3. **Marginals** tell you *which axis to spend money on* for other tessera
   variants later (e.g. if lat dim dominates and aux is a wash, future ablations
   can fix aux=off).
4. **Pending**: 2024 `crop128_lat32_auxoff` fills in automatically once its VAE
   finishes → copy the latents (see `processed/vae_tessera_1B-M/provenance.txt`)
   and re-run the two 2024 submit scripts (idempotent, +12 jobs).
5. **Rollout**: once a winner is chosen, clone the four experiment folders'
   pattern to `us` / `australia` / `southern_africa` with only the winning
   variant's entries (or reuse the full yaml and submit selectively).